# Review references

This notebook will audit validation and test commit subjects against their diffs and explore the prepared data.

## Import dependencies

In [ ]:
from collections import Counter
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from whats_that_code.election import guess_language_all_methods

## Configure review paths

Set the directory containing the prepared validation and test splits and the private sidecar location for review decisions.

In [ ]:
prepared_directory = Path("prepared")
review_path = prepared_directory / "review.parquet"

## Load the split datasets

This block loads the prepared validation and test Parquet files.

In [ ]:
validation_path = prepared_directory / "validation.parquet"
test_path = prepared_directory / "test.parquet"
if not validation_path.is_file() or not test_path.is_file():
    raise FileNotFoundError("Prepared validation and test Parquet files are required.")

validation_table = pq.read_table(validation_path)
test_table = pq.read_table(test_path)

## Validate the reference columns

The review needs an identifier, subject, diff, commit date, and repository group. This block checks that both held-out splits contain those columns and converts them into rows.

In [ ]:
reference_columns = (
    "id",
    "repository_group_id",
    "subject",
    "diff",
    "committed_at_utc",
)
for split_name, table in (("validation", validation_table), ("test", test_table)):
    missing_columns = [column for column in reference_columns if column not in table.column_names]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {', '.join(missing_columns)}")

validation_rows = validation_table.select(list(reference_columns)).to_pylist()
test_rows = test_table.select(list(reference_columns)).to_pylist()

## Explore the held-out references

This block summarizes subject and diff lengths, date ranges, repository groups, and detected languages.

In [ ]:
def detect_languages(rows: list[dict[str, object]]) -> dict[str, int]:
    counts = Counter()
    for row in rows:
        for line in str(row["diff"]).splitlines():
            if line.startswith("+++ b/"):
                language = guess_language_all_methods(str(row["diff"]), file_name=line[6:])
                counts[language or "unknown"] += 1
    return dict(counts)


def split_summary(rows: list[dict[str, object]]) -> dict[str, object]:
    subjects = [str(row["subject"]) for row in rows]
    diffs = [str(row["diff"]) for row in rows]
    return {
        "rows": len(rows),
        "subject_length": {"min": min(map(len, subjects)), "max": max(map(len, subjects))},
        "diff_length": {"min": min(map(len, diffs)), "max": max(map(len, diffs))},
        "date_range": {
            "first": min(str(row["committed_at_utc"]) for row in rows),
            "last": max(str(row["committed_at_utc"]) for row in rows),
        },
        "repository_groups": len({row["repository_group_id"] for row in rows}),
        "languages": detect_languages(rows),
    }


eda_summary = {
    "validation": split_summary(validation_rows),
    "test": split_summary(test_rows),
}
eda_summary

## Build the audit queue

This block combines validation and test references into the queue that will be reviewed. Each entry keeps only the split, identifier, subject, and diff needed for the audit.

In [ ]:
audit_queue = [
    {"split": split_name, "id": row["id"], "subject": row["subject"], "diff": row["diff"]}
    for split_name, rows in (("validation", validation_rows), ("test", test_rows))
    for row in rows
]
len(audit_queue)

## Load the review-only Qwen model

This loads the untouched base model locally from Hugging Face on the WSL ROCm GPU. It is used only to summarize diffs for this review notebook; it is not trained, modified, or included in the project package. The first run downloads model weights, then Hugging Face reuses its local cache.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("The ROCm GPU is unavailable. Select the Qwen Commit Review (ROCm) kernel.")

review_model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
review_device = torch.device("cuda")
review_tokenizer = AutoTokenizer.from_pretrained(review_model_id)
review_model = AutoModelForCausalLM.from_pretrained(review_model_id, torch_dtype=torch.bfloat16)
review_model.to(review_device)
review_model.eval()
print(torch.cuda.get_device_name())

## Summarize a reference with Qwen

This function asks the untouched local model for a short factual digest. It truncates the diff during tokenization to fit the GPU prompt budget and states when content may have been omitted. The model does not approve or reject the reference.

In [ ]:
review_max_diff_tokens = 4_096
review_max_new_tokens = 96


def summarize_reference(reference: dict[str, object]) -> str:
    diff_ids = review_tokenizer(
        str(reference["diff"]),
        add_special_tokens=False,
        truncation=True,
        max_length=review_max_diff_tokens,
    )["input_ids"]
    diff_truncated = len(diff_ids) == review_max_diff_tokens
    digest_diff = review_tokenizer.decode(diff_ids, skip_special_tokens=True)
    messages = [
        {
            "role": "system",
            "content": (
                "Summarize a Git diff for a human reviewing a historical commit subject. "
                "Be factual and concise. Do not decide whether to approve or reject the subject."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Historical subject: {reference['subject']}\n\n"
                f"Diff:\n{digest_diff}\n\n"
                "Return exactly three bullets: main change, affected areas, and whether the subject "
                "appears accurate, vague, or mismatched with a brief reason."
            ),
        },
    ]
    prompt = review_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = review_tokenizer(prompt, return_tensors="pt").to(review_device)
    with torch.inference_mode():
        generated_ids = review_model.generate(
            **model_inputs, do_sample=False, max_new_tokens=review_max_new_tokens
        )
    completion_ids = generated_ids[0][model_inputs.input_ids.shape[-1] :]
    summary = review_tokenizer.decode(completion_ids, skip_special_tokens=True).strip()
    if diff_truncated:
        summary = f"Diff truncated to {review_max_diff_tokens} tokens.\n{summary}"
    return summary

## Review the references

Validation is reviewed first, followed by test. Each approve or reject decision is written to the private sidecar immediately. Enter a to approve a reference, r to reject it, d to inspect the full diff, or q to stop; rerunning this block resumes from the sidecar. The audit stops automatically after 50 approvals in each split.

In [ ]:
required_approvals = 50
review_schema = pa.schema(
    [
        pa.field("id", pa.string()),
        pa.field("split", pa.string()),
        pa.field("approved", pa.bool_()),
    ]
)
references_by_id = {reference["id"]: reference for reference in audit_queue}

if review_path.is_file():
    existing_reviews = pq.read_table(review_path).to_pylist()
    review_decisions = {row["id"]: row["approved"] for row in existing_reviews}
else:
    review_decisions = {}


def save_review_decisions() -> None:
    review_rows = [
        {
            "id": reference["id"],
            "split": reference["split"],
            "approved": review_decisions[reference["id"]],
        }
        for reference in audit_queue
        if reference["id"] in review_decisions
    ]
    pq.write_table(
        pa.Table.from_pylist(review_rows, schema=review_schema),
        review_path,
        compression="zstd",
    )


def approved_count(split_name: str) -> int:
    return sum(
        review_decisions.get(reference["id"], False)
        for reference in audit_queue
        if reference["split"] == split_name
    )


quit_requested = False
for split_name in ("validation", "test"):
    while approved_count(split_name) < required_approvals:
        reference = next(
            (
                reference
                for reference in audit_queue
                if reference["split"] == split_name and reference["id"] not in review_decisions
            ),
            None,
        )
        if reference is None:
            raise RuntimeError(
                f"{split_name} ran out of references before reaching "
                f"{required_approvals} approvals."
            )

        print(f"[{split_name} {approved_count(split_name)}/{required_approvals}] {reference['id']}")
        print(f"Subject: {reference['subject']}")
        print(f"Qwen digest:\n{summarize_reference(reference)}")
        while True:
            decision = input("Decision [a]pprove/[r]eject/[d]iff/[q]uit: ").strip().casefold()
            if decision in {"a", "r"}:
                review_decisions[reference["id"]] = decision == "a"
                save_review_decisions()
                break
            if decision == "d":
                print(f"Diff:\n{reference['diff']}")
                continue
            if decision == "q":
                break
            print("Enter a, r, d, or q.")
        if decision == "q":
            quit_requested = True
            break
    if quit_requested:
        break

{split_name: approved_count(split_name) for split_name in ("validation", "test")}

## Materialize reviewed splits

This block materializes approved validation and test references into their own Parquet files, then removes the checkpoint sidecar. The original prepared split files remain unchanged.

In [ ]:
if not review_path.is_file():
    raise FileNotFoundError("No review sidecar exists yet. Finish the review block first.")

review_rows = pq.read_table(review_path).to_pylist()
approved_ids_by_split = {
    split_name: {row["id"] for row in review_rows if row["split"] == split_name and row["approved"]}
    for split_name in ("validation", "test")
}
for split_name, approved_ids in approved_ids_by_split.items():
    if len(approved_ids) < required_approvals:
        raise ValueError(
            f"{split_name} needs at least {required_approvals} approved references; "
            f"found {len(approved_ids)}."
        )

source_tables = {
    "validation": validation_table,
    "test": test_table,
}
reviewed_tables = {}
for split_name, source_table in source_tables.items():
    approved_ids = approved_ids_by_split[split_name]
    source_ids = set(source_table["id"].to_pylist())
    missing_ids = approved_ids - source_ids
    if missing_ids:
        raise ValueError(
            f"{split_name} review sidecar contains {len(missing_ids)} ID(s) absent from "
            "the current prepared split. Re-run the review for the regenerated data."
        )

    reviewed_table = source_table.filter(
        pc.is_in(
            source_table["id"],
            value_set=pa.array(sorted(approved_ids)),
        )
    )
    if reviewed_table.num_rows < required_approvals:
        raise RuntimeError(
            f"{split_name} has only {reviewed_table.num_rows} reviewed rows after filtering."
        )
    reviewed_tables[split_name] = reviewed_table

reviewed_paths = {}
for split_name, reviewed_table in reviewed_tables.items():
    reviewed_path = prepared_directory / f"reviewed_{split_name}.parquet"
    pq.write_table(reviewed_table, reviewed_path, compression="zstd")
    reviewed_paths[split_name] = reviewed_path

review_path.unlink()
{
    split_name: {
        "rows": pq.read_metadata(path).num_rows,
        "path": str(path),
    }
    for split_name, path in reviewed_paths.items()
}